In [6]:
"""
HinEmo — Complete Annotation Validation Report
================================================
Computes everything needed for the annotation validation section of the paper:
- Overall Cohen's Kappa
- Per-emotion Kappa  
- Raw agreement rate
- Confusion matrix
- Disagreement breakdown by pair
- Error categorization
- Representative disagreement examples per category
"""

import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# ── 1. LOAD FILES ──────────────────────────────────────────────────────────────

manual = pd.read_csv(
    "/Users/harshaggarwal/Projects_4/hinemo_project/data/manual_relabeling_samples/Final Manually Relabelled.csv",
    encoding="utf-8"
)
print(f"Manual file columns: {manual.columns.tolist()}")
print(f"Manual file shape: {manual.shape}")

merged = pd.read_csv(
    "/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/hinemo_dataset_draft_1(46174).csv"
)
print(f"Merged file columns: {merged.columns.tolist()}")
print(f"Merged file shape: {merged.shape}")

# ── 2. RENAME COLUMNS TO STANDARD NAMES ───────────────────────────────────────

manual = manual.rename(columns={"manual emotion(joy,sadness,anger disgust)": "manual_label"})
manual["id"] = manual["id"].astype(str)
merged["id"] = merged["id"].astype(str)

# ── 3. JOIN ON ID ──────────────────────────────────────────────────────────────

combined = manual.merge(
    merged[["id", "gpt_emotion", "text"]],
    on="id",
    how="inner"
)
combined = combined.rename(columns={"gpt_emotion": "gpt_label"})

print(f"\nTotal manual samples:   {len(manual)}")
print(f"Successfully matched:   {len(combined)}")
print(f"Unmatched:              {len(manual) - len(combined)}")

# ── 4. DROP UNANNOTATED ROWS ───────────────────────────────────────────────────

blanks = combined[combined["manual_label"].isna()]
print(f"Blank manual labels:    {len(blanks)}")
if len(blanks) > 0:
    print("Breakdown of blanks by source and GPT label:")
    print(blanks[["source", "gpt_label"]].value_counts())

combined_clean = combined.dropna(subset=["manual_label"]).copy()
combined_clean["manual_label"] = combined_clean["manual_label"].str.strip().str.lower()
combined_clean["gpt_label"] = combined_clean["gpt_label"].str.strip().str.lower()
print(f"Final sample for kappa: {len(combined_clean)}")

# ── 5. LABEL DISTRIBUTIONS ─────────────────────────────────────────────────────

print("\n" + "="*60)
print("LABEL DISTRIBUTIONS")
print("="*60)
dist = pd.DataFrame({
    "manual_count": combined_clean["manual_label"].value_counts(),
    "gpt_count":    combined_clean["gpt_label"].value_counts(),
})
dist["manual_pct"] = (dist["manual_count"] / len(combined_clean) * 100).round(1)
dist["gpt_pct"]    = (dist["gpt_count"]    / len(combined_clean) * 100).round(1)
print(dist)

# ── 6. RAW AGREEMENT RATE ──────────────────────────────────────────────────────

agreement = (combined_clean["manual_label"] == combined_clean["gpt_label"]).mean()
n_disagree = (combined_clean["manual_label"] != combined_clean["gpt_label"]).sum()
print(f"\nRaw agreement rate:  {agreement:.1%}")
print(f"Total disagreements: {n_disagree}")

# ── 7. OVERALL COHEN'S KAPPA ───────────────────────────────────────────────────

print("\n" + "="*60)
print("COHEN'S KAPPA")
print("="*60)

overall_kappa = cohen_kappa_score(
    combined_clean["manual_label"],
    combined_clean["gpt_label"]
)
print(f"Overall Cohen's Kappa: {overall_kappa:.4f}")

if overall_kappa >= 0.80:
    interp = "Almost perfect agreement"
elif overall_kappa >= 0.60:
    interp = "Substantial agreement"
elif overall_kappa >= 0.40:
    interp = "Moderate agreement"
elif overall_kappa >= 0.20:
    interp = "Fair agreement"
else:
    interp = "Poor agreement — investigate"
print(f"Interpretation:        {interp}")

# ── 8. PER-EMOTION KAPPA ───────────────────────────────────────────────────────

print("\nPer-emotion Kappa (one-vs-rest binary):")
emotion_kappas = {}
for emotion in ["joy", "anger", "sadness", "disgust"]:
    binary_manual = (combined_clean["manual_label"] == emotion).astype(int)
    binary_gpt    = (combined_clean["gpt_label"]    == emotion).astype(int)
    k = cohen_kappa_score(binary_manual, binary_gpt)
    emotion_kappas[emotion] = k
    bar = "█" * int(k * 20)
    print(f"  {emotion:<10} κ = {k:.4f}  {bar}")

# ── 9. CONFUSION MATRIX ────────────────────────────────────────────────────────

print("\n" + "="*60)
print("CONFUSION MATRIX  (rows = human label, cols = GPT label)")
print("="*60)
labels = ["anger", "disgust", "joy", "sadness"]
cm = confusion_matrix(
    combined_clean["manual_label"],
    combined_clean["gpt_label"],
    labels=labels
)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

print("\nPer-class stats:")
print(f"{'emotion':<10} {'human_total':>12} {'gpt_predicted':>14} {'exact_match':>12} {'hit_rate':>10}")
for i, emotion in enumerate(labels):
    human_total   = cm[i].sum()
    gpt_predicted = cm[:, i].sum()
    exact_match   = cm[i][i]
    hit_rate      = exact_match / human_total if human_total > 0 else 0
    print(f"{emotion:<10} {human_total:>12} {gpt_predicted:>14} {exact_match:>12} {hit_rate:>9.1%}")

# ── 10. DISAGREEMENT PAIR BREAKDOWN ────────────────────────────────────────────

print("\n" + "="*60)
print("DISAGREEMENT PAIRS  (human → GPT)")
print("="*60)
disagreements = combined_clean[
    combined_clean["manual_label"] != combined_clean["gpt_label"]
].copy()

pair_counts = (
    disagreements
    .groupby(["manual_label", "gpt_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
pair_counts["pct_of_disagreements"] = (
    pair_counts["count"] / len(disagreements) * 100
).round(1)
pair_counts["pct_of_total"] = (
    pair_counts["count"] / len(combined_clean) * 100
).round(1)
print(pair_counts.to_string(index=False))

# ── 11. ERROR CATEGORIZATION ───────────────────────────────────────────────────

print("\n" + "="*60)
print("ERROR CATEGORIZATION")
print("="*60)

def categorize_disagreement(manual, gpt):
    pair = tuple(sorted([manual, gpt]))
    if pair == ("anger", "disgust"):
        return "anger-disgust boundary"
    elif pair in [("anger", "sadness"), ("disgust", "sadness")]:
        return "negative-emotion boundary"
    elif "joy" in pair:
        return "positive-negative crossing (likely sarcasm misfire)"
    else:
        return "other"

disagreements["error_category"] = disagreements.apply(
    lambda r: categorize_disagreement(r["manual_label"], r["gpt_label"]),
    axis=1
)

cat_counts = disagreements["error_category"].value_counts()
cat_pcts   = (disagreements["error_category"].value_counts(normalize=True) * 100).round(1)
cat_df = pd.DataFrame({
    "count": cat_counts,
    "pct_of_disagreements": cat_pcts,
    "pct_of_total": (cat_counts / len(combined_clean) * 100).round(1)
})
print(cat_df.to_string())

# ── 12. REPRESENTATIVE EXAMPLES PER ERROR CATEGORY ────────────────────────────

print("\n" + "="*60)
print("REPRESENTATIVE DISAGREEMENT EXAMPLES")
print("="*60)

for category in disagreements["error_category"].unique():
    subset = disagreements[disagreements["error_category"] == category]
    print(f"\n--- {category} ({len(subset)} cases) ---")
    sample = subset.sample(min(5, len(subset)), random_state=42)
    for _, row in sample.iterrows():
        text = str(row.get("text", "")) if pd.notna(row.get("text", "")) else "(no text)"
        print(f"  [human={row['manual_label']}, GPT={row['gpt_label']}]")
        print(f"  {text[:150]}")
        print()

# ── 13. KAPPA WITHOUT ANGER-DISGUST BOUNDARY CASES ────────────────────────────

print("\n" + "="*60)
print("KAPPA EXCLUDING ANGER-DISGUST BOUNDARY CASES")
print("="*60)

# Remove rows where both labels are in {anger, disgust} but they disagree
anger_disgust_mask = (
    (combined_clean["manual_label"].isin(["anger", "disgust"])) &
    (combined_clean["gpt_label"].isin(["anger", "disgust"])) &
    (combined_clean["manual_label"] != combined_clean["gpt_label"])
)
filtered = combined_clean[~anger_disgust_mask]
filtered_kappa = cohen_kappa_score(
    filtered["manual_label"],
    filtered["gpt_label"]
)
print(f"Rows removed (anger↔disgust disagreements): {anger_disgust_mask.sum()}")
print(f"Remaining rows: {len(filtered)}")
print(f"Kappa excluding anger↔disgust boundary: {filtered_kappa:.4f}")
print(f"(vs. overall κ = {overall_kappa:.4f} — improvement of {filtered_kappa - overall_kappa:.4f})")

# ── 14. FINAL SUMMARY ─────────────────────────────────────────────────────────

print("\n" + "="*60)
print("PAPER-READY SUMMARY")
print("="*60)
ad_count = cat_counts.get("anger-disgust boundary", 0)
print(f"""
Sample: {len(combined_clean)} comments ({len(blanks)} unannotated excluded from {len(manual)} total)

Overall Cohen's κ     = {overall_kappa:.4f}  ({interp})
Excluding anger↔disgust boundary cases: κ = {filtered_kappa:.4f}

Per-emotion κ:
  joy:     {emotion_kappas['joy']:.4f}
  anger:   {emotion_kappas['anger']:.4f}
  sadness: {emotion_kappas['sadness']:.4f}
  disgust: {emotion_kappas['disgust']:.4f}

Raw agreement rate: {agreement:.1%}
Total disagreements: {n_disagree} ({n_disagree/len(combined_clean):.1%})

Of {n_disagree} disagreements:
  {ad_count} ({ad_count/n_disagree*100:.1f}%) are anger↔disgust boundary cases
  — consistent with their theoretical proximity in Plutchik's wheel
  — removing these raises κ from {overall_kappa:.4f} to {filtered_kappa:.4f}
  — suggesting label quality on the remaining three classes is substantially higher
""")

Manual file columns: ['id', 'text', 'manual emotion(joy,sadness,anger disgust)', 'source']
Manual file shape: (1103, 4)
Merged file columns: ['id', 'text', 'gpt_emotion', 'source']
Merged file shape: (46174, 4)

Total manual samples:   1103
Successfully matched:   1103
Unmatched:              0
Blank manual labels:    12
Breakdown of blanks by source and GPT label:
source          gpt_label
youtube_round1  disgust      3
youtube_round2  disgust      2
youtube_round1  joy          2
                anger        1
sentimix        joy          1
teaser5k        disgust      1
youtube_round2  anger        1
teaser5k        joy          1
Name: count, dtype: int64
Final sample for kappa: 1091

LABEL DISTRIBUTIONS
         manual_count  gpt_count  manual_pct  gpt_pct
anger             333        291        30.5     26.7
disgust           117        194        10.7     17.8
joy               464        437        42.5     40.1
sadness           177        169        16.2     15.5

Raw agreeme

In [7]:
combined = manual.merge(
    merged[["id", "gpt_emotion", "text"]].rename(columns={"text": "text_gpt"}),
    on="id",
    how="inner"
)
combined = combined.rename(columns={"gpt_emotion": "gpt_label"})

In [9]:
"""
HinEmo — Complete Annotation Validation Report
================================================
Computes everything needed for the annotation validation section of the paper:
- Overall Cohen's Kappa
- Per-emotion Kappa  
- Raw agreement rate
- Confusion matrix
- Disagreement breakdown by pair
- Error categorization
- Representative disagreement examples per category
"""

import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# ── 1. LOAD FILES ──────────────────────────────────────────────────────────────

manual = pd.read_csv(
    "/Users/harshaggarwal/Projects_4/hinemo_project/data/manual_relabeling_samples/Final Manually Relabelled.csv",
    encoding="utf-8"
)
print(f"Manual file columns: {manual.columns.tolist()}")
print(f"Manual file shape: {manual.shape}")

merged = pd.read_csv(
    "/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/hinemo_dataset_draft_1(46174).csv"
)
print(f"Merged file columns: {merged.columns.tolist()}")
print(f"Merged file shape: {merged.shape}")

# ── 2. RENAME COLUMNS TO STANDARD NAMES ───────────────────────────────────────

manual = manual.rename(columns={"manual emotion(joy,sadness,anger disgust)": "manual_label"})
manual["id"] = manual["id"].astype(str)
merged["id"] = merged["id"].astype(str)

# ── 3. JOIN ON ID ──────────────────────────────────────────────────────────────

combined = manual.merge(
    merged[["id", "gpt_emotion", "text"]],
    on="id",
    how="inner"
)
combined = combined.rename(columns={"gpt_emotion": "gpt_label"})

print(f"\nTotal manual samples:   {len(manual)}")
print(f"Successfully matched:   {len(combined)}")
print(f"Unmatched:              {len(manual) - len(combined)}")

# ── 4. DROP UNANNOTATED ROWS ───────────────────────────────────────────────────

blanks = combined[combined["manual_label"].isna()]
print(f"Blank manual labels:    {len(blanks)}")
if len(blanks) > 0:
    print("Breakdown of blanks by source and GPT label:")
    print(blanks[["source", "gpt_label"]].value_counts())

combined_clean = combined.dropna(subset=["manual_label"]).copy()
combined_clean["manual_label"] = combined_clean["manual_label"].str.strip().str.lower()
combined_clean["gpt_label"] = combined_clean["gpt_label"].str.strip().str.lower()
print(f"Final sample for kappa: {len(combined_clean)}")

# ── 5. LABEL DISTRIBUTIONS ─────────────────────────────────────────────────────

print("\n" + "="*60)
print("LABEL DISTRIBUTIONS")
print("="*60)
dist = pd.DataFrame({
    "manual_count": combined_clean["manual_label"].value_counts(),
    "gpt_count":    combined_clean["gpt_label"].value_counts(),
})
dist["manual_pct"] = (dist["manual_count"] / len(combined_clean) * 100).round(1)
dist["gpt_pct"]    = (dist["gpt_count"]    / len(combined_clean) * 100).round(1)
print(dist)

# ── 6. RAW AGREEMENT RATE ──────────────────────────────────────────────────────

agreement = (combined_clean["manual_label"] == combined_clean["gpt_label"]).mean()
n_disagree = (combined_clean["manual_label"] != combined_clean["gpt_label"]).sum()
print(f"\nRaw agreement rate:  {agreement:.1%}")
print(f"Total disagreements: {n_disagree}")

# ── 7. OVERALL COHEN'S KAPPA ───────────────────────────────────────────────────

print("\n" + "="*60)
print("COHEN'S KAPPA")
print("="*60)

overall_kappa = cohen_kappa_score(
    combined_clean["manual_label"],
    combined_clean["gpt_label"]
)
print(f"Overall Cohen's Kappa: {overall_kappa:.4f}")

if overall_kappa >= 0.80:
    interp = "Almost perfect agreement"
elif overall_kappa >= 0.60:
    interp = "Substantial agreement"
elif overall_kappa >= 0.40:
    interp = "Moderate agreement"
elif overall_kappa >= 0.20:
    interp = "Fair agreement"
else:
    interp = "Poor agreement — investigate"
print(f"Interpretation:        {interp}")

# ── 8. PER-EMOTION KAPPA ───────────────────────────────────────────────────────

print("\nPer-emotion Kappa (one-vs-rest binary):")
emotion_kappas = {}
for emotion in ["joy", "anger", "sadness", "disgust"]:
    binary_manual = (combined_clean["manual_label"] == emotion).astype(int)
    binary_gpt    = (combined_clean["gpt_label"]    == emotion).astype(int)
    k = cohen_kappa_score(binary_manual, binary_gpt)
    emotion_kappas[emotion] = k
    bar = "█" * int(k * 20)
    print(f"  {emotion:<10} κ = {k:.4f}  {bar}")

# ── 9. CONFUSION MATRIX ────────────────────────────────────────────────────────

print("\n" + "="*60)
print("CONFUSION MATRIX  (rows = human label, cols = GPT label)")
print("="*60)
labels = ["anger", "disgust", "joy", "sadness"]
cm = confusion_matrix(
    combined_clean["manual_label"],
    combined_clean["gpt_label"],
    labels=labels
)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

print("\nPer-class stats:")
print(f"{'emotion':<10} {'human_total':>12} {'gpt_predicted':>14} {'exact_match':>12} {'hit_rate':>10}")
for i, emotion in enumerate(labels):
    human_total   = cm[i].sum()
    gpt_predicted = cm[:, i].sum()
    exact_match   = cm[i][i]
    hit_rate      = exact_match / human_total if human_total > 0 else 0
    print(f"{emotion:<10} {human_total:>12} {gpt_predicted:>14} {exact_match:>12} {hit_rate:>9.1%}")

# ── 10. DISAGREEMENT PAIR BREAKDOWN ────────────────────────────────────────────

print("\n" + "="*60)
print("DISAGREEMENT PAIRS  (human → GPT)")
print("="*60)
disagreements = combined_clean[
    combined_clean["manual_label"] != combined_clean["gpt_label"]
].copy()

pair_counts = (
    disagreements
    .groupby(["manual_label", "gpt_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
pair_counts["pct_of_disagreements"] = (
    pair_counts["count"] / len(disagreements) * 100
).round(1)
pair_counts["pct_of_total"] = (
    pair_counts["count"] / len(combined_clean) * 100
).round(1)
print(pair_counts.to_string(index=False))

# ── 11. ERROR CATEGORIZATION ───────────────────────────────────────────────────

print("\n" + "="*60)
print("ERROR CATEGORIZATION")
print("="*60)

def categorize_disagreement(manual, gpt):
    pair = tuple(sorted([manual, gpt]))
    if pair == ("anger", "disgust"):
        return "anger-disgust boundary"
    elif pair in [("anger", "sadness"), ("disgust", "sadness")]:
        return "negative-emotion boundary"
    elif "joy" in pair:
        return "positive-negative crossing (likely sarcasm misfire)"
    else:
        return "other"

disagreements["error_category"] = disagreements.apply(
    lambda r: categorize_disagreement(r["manual_label"], r["gpt_label"]),
    axis=1
)

cat_counts = disagreements["error_category"].value_counts()
cat_pcts   = (disagreements["error_category"].value_counts(normalize=True) * 100).round(1)
cat_df = pd.DataFrame({
    "count": cat_counts,
    "pct_of_disagreements": cat_pcts,
    "pct_of_total": (cat_counts / len(combined_clean) * 100).round(1)
})
print(cat_df.to_string())

# ── 12. REPRESENTATIVE EXAMPLES PER ERROR CATEGORY ────────────────────────────

print("\n" + "="*60)
print("REPRESENTATIVE DISAGREEMENT EXAMPLES")
print("="*60)

for category in disagreements["error_category"].unique():
    subset = disagreements[disagreements["error_category"] == category]
    print(f"\n--- {category} ({len(subset)} cases) ---")
    sample = subset.sample(min(5, len(subset)), random_state=42)
    for _, row in sample.iterrows():
        text = str(row.get("text_gpt", "")) if pd.notna(row.get("text_gpt", "")) else "(no text)"
        print(f"  [human={row['manual_label']}, GPT={row['gpt_label']}]")
        print(f"  {text[:150]}")
        print()

# ── 13. KAPPA WITHOUT ANGER-DISGUST BOUNDARY CASES ────────────────────────────

print("\n" + "="*60)
print("KAPPA EXCLUDING ANGER-DISGUST BOUNDARY CASES")
print("="*60)

# Remove rows where both labels are in {anger, disgust} but they disagree
anger_disgust_mask = (
    (combined_clean["manual_label"].isin(["anger", "disgust"])) &
    (combined_clean["gpt_label"].isin(["anger", "disgust"])) &
    (combined_clean["manual_label"] != combined_clean["gpt_label"])
)
filtered = combined_clean[~anger_disgust_mask]
filtered_kappa = cohen_kappa_score(
    filtered["manual_label"],
    filtered["gpt_label"]
)
print(f"Rows removed (anger↔disgust disagreements): {anger_disgust_mask.sum()}")
print(f"Remaining rows: {len(filtered)}")
print(f"Kappa excluding anger↔disgust boundary: {filtered_kappa:.4f}")
print(f"(vs. overall κ = {overall_kappa:.4f} — improvement of {filtered_kappa - overall_kappa:.4f})")

# ── 14. FINAL SUMMARY ─────────────────────────────────────────────────────────

print("\n" + "="*60)
print("PAPER-READY SUMMARY")
print("="*60)
ad_count = cat_counts.get("anger-disgust boundary", 0)
print(f"""
Sample: {len(combined_clean)} comments ({len(blanks)} unannotated excluded from {len(manual)} total)

Overall Cohen's κ     = {overall_kappa:.4f}  ({interp})
Excluding anger↔disgust boundary cases: κ = {filtered_kappa:.4f}

Per-emotion κ:
  joy:     {emotion_kappas['joy']:.4f}
  anger:   {emotion_kappas['anger']:.4f}
  sadness: {emotion_kappas['sadness']:.4f}
  disgust: {emotion_kappas['disgust']:.4f}

Raw agreement rate: {agreement:.1%}
Total disagreements: {n_disagree} ({n_disagree/len(combined_clean):.1%})

Of {n_disagree} disagreements:
  {ad_count} ({ad_count/n_disagree*100:.1f}%) are anger↔disgust boundary cases
  — consistent with their theoretical proximity in Plutchik's wheel
  — removing these raises κ from {overall_kappa:.4f} to {filtered_kappa:.4f}
  — suggesting label quality on the remaining three classes is substantially higher
""")

Manual file columns: ['id', 'text', 'manual emotion(joy,sadness,anger disgust)', 'source']
Manual file shape: (1103, 4)
Merged file columns: ['id', 'text', 'gpt_emotion', 'source']
Merged file shape: (46174, 4)

Total manual samples:   1103
Successfully matched:   1103
Unmatched:              0
Blank manual labels:    12
Breakdown of blanks by source and GPT label:
source          gpt_label
youtube_round1  disgust      3
youtube_round2  disgust      2
youtube_round1  joy          2
                anger        1
sentimix        joy          1
teaser5k        disgust      1
youtube_round2  anger        1
teaser5k        joy          1
Name: count, dtype: int64
Final sample for kappa: 1091

LABEL DISTRIBUTIONS
         manual_count  gpt_count  manual_pct  gpt_pct
anger             333        291        30.5     26.7
disgust           117        194        10.7     17.8
joy               464        437        42.5     40.1
sadness           177        169        16.2     15.5

Raw agreeme